# 🌊 Project 9 — Calculate NDWI (Water Map)

## 🎯 Goal

In this project, you will learn how to detect water surfaces using satellite imagery.

You will create maps showing:

- 🌊 Lakes
- 💧 Rivers
- 🚰 Reservoirs
- 🌍 Water bodies

You will learn:

- What NDWI is.
- Why water has a different spectral response.
- How to create a water mask.

---

# 📚 Skills You'll Learn

- ✅ Spectral indices
- ✅ NDWI calculation
- ✅ Water detection
- ✅ Thresholding
- ✅ Masking

---

# 🌍 Why Is Water Mapping Important?

Water detection is one of the most common remote sensing applications.

It is used for:

- 🌊 Lake and reservoir monitoring
- 💧 Flood mapping
- 🚰 Water resource management
- 🌍 Climate change studies
- 🛰️ Drought analysis

Examples:

A researcher may ask:

- How has a lake changed over 20 years?
- Which areas were flooded after heavy rainfall?
- Is a reservoir shrinking because of drought?

Satellite imagery helps answer these questions.

---

# 🌱 Different Surfaces Need Different Spectral Indices

You already learned NDVI.

For vegetation:

```text
NDVI

B8 (NIR)
+
B4 (Red)
```

For water:

```text
NDWI

B3 (Green)
+
B8 (NIR)
```

The workflow is the same.

Only the bands change.

---

# Step 1 — Understand NDWI

## NDVI Reminder

The vegetation index:

```text
        NIR - Red
NDVI = -------------
        NIR + Red
```

Sentinel-2:

```text
NIR = B8

Red = B4
```

---

## NDWI Formula

The water index:

```text
        Green - NIR
NDWI = --------------
        Green + NIR
```

Sentinel-2:

```text
Green = B3

NIR = B8
```

In Earth Engine:

```javascript
normalizedDifference(['B3','B8'])
```

---

# Why Does NDWI Detect Water?

Different surfaces reflect different wavelengths.

## 🌊 Water

Water:

- Reflects some Green light.
- Absorbs Near Infrared strongly.

Therefore:

```text
Green ↑

NIR ↓
```

Result:

```text
NDWI > 0
```

---

## 🌿 Vegetation

Vegetation:

- Reflects Near Infrared strongly.
- Reflects less Green compared to NIR.

Therefore:

```text
NIR ↑↑
```

Result:

```text
NDWI usually < 0
```

---

# Step 2 — Create Study Area

We choose an area containing water.

Example: Berlin lakes.

```javascript
var cityArea = ee.Geometry.Polygon([
  [
    [13.20,52.60],
    [13.65,52.60],
    [13.65,52.35],
    [13.20,52.35],
    [13.20,52.60]
  ]
]);
```

---

# Step 3 — Load Sentinel-2 Image

Same workflow as previous projects.

```javascript
var image = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(cityArea)
.filterDate('2025-06-01','2025-08-31')
.sort('CLOUDY_PIXEL_PERCENTAGE')
.first()
.clip(cityArea);
```

Why summer?

Because:

- Fewer clouds.
- Clearer water.
- Easier interpretation.

---

# Step 4 — Display True Color

First, let's look at the original satellite image.

```javascript
var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};


Map.centerObject(cityArea,10);


Map.addLayer(
  image,
  trueColor,
  'True Color'
);
```

---

# Step 5 — Calculate NDWI ⭐

Now we create the water index.

```javascript
var ndwi = image
.normalizedDifference(['B3','B8'])
.rename('NDWI');
```

Let's break it down.

---

## Input

```javascript
image
```

The Sentinel-2 image.

↓

## Function

```javascript
normalizedDifference()
```

Calculates:

```text
Band 1 - Band 2
----------------
Band 1 + Band 2
```

↓

## Bands

```javascript
['B3','B8']
```

Means:

```text
Green - NIR
```

↓

## Output

A new image:

Before:

```text
Sentinel-2 Image

B1
B2
B3
...
B12
```

After:

```text
NDWI Image

NDWI
```

Every pixel now contains an NDWI value.

---

# Step 6 — NDWI Visualization

NDWI values usually range:

```text
-1 ------------ 0 ------------ +1
```

Typical interpretation:

| NDWI | Meaning |
|---|---|
| Negative | Land |
| Around 0 | Mixed surfaces |
| Positive | Water |

Visualization:

```javascript
var ndwiVis = {
  min:-1,
  max:1,
  palette:[
    'brown',
    'white',
    'blue'
  ]
};
```

Color meaning:

| Color | Meaning |
|---|---|
| Brown | Land |
| White | Mixed pixels |
| Blue | Water |

---

# Step 7 — Display NDWI

```javascript
Map.addLayer(
  ndwi,
  ndwiVis,
  'NDWI'
);
```

Now you have a water index map.

---

# Step 8 — Create Water Mask ⭐

Similar to Project 8.

Question:

Which pixels are water?

We define:

```text
Water = NDWI > 0
```

Code:

```javascript
var waterMask = ndwi.gt(0);
```

The result:

```text
NDWI Image

↓

Threshold

↓

Boolean Image
```

Example:

```text
0.5  → True

0.2  → True

-0.3 → False
```

---

# Step 9 — Display Water Only

Apply the mask:

```javascript
var waterMap = ndwi.updateMask(waterMask);
```

Now display:

```javascript
Map.addLayer(
  waterMap,
  {
    palette:['blue']
  },
  'Water'
);
```

Only water pixels remain visible.

---

# Complete Code

```javascript
// Study Area
var cityArea = ee.Geometry.Polygon([
  [
    [13.20,52.60],
    [13.65,52.60],
    [13.65,52.35],
    [13.20,52.35],
    [13.20,52.60]
  ]
]);


// Load Image
var image = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(cityArea)
.filterDate('2025-06-01','2025-08-31')
.sort('CLOUDY_PIXEL_PERCENTAGE')
.first()
.clip(cityArea);


// True Color
var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};


Map.centerObject(cityArea,10);


Map.addLayer(
  image,
  trueColor,
  'True Color'
);


// NDWI
var ndwi = image
.normalizedDifference(['B3','B8'])
.rename('NDWI');


// NDWI Visualization
var ndwiVis = {
  min:-1,
  max:1,
  palette:[
    'brown',
    'white',
    'blue'
  ]
};


Map.addLayer(
  ndwi,
  ndwiVis,
  'NDWI'
);


// Water Mask
var waterMask = ndwi.gt(0);


// Water Map
var waterMap = ndwi.updateMask(waterMask);


Map.addLayer(
  waterMap,
  {
    palette:['blue']
  },
  'Water'
);
```

---

# Important Concepts

## 1. NDVI and NDWI Have the Same Structure

Both follow:

```text
Satellite Image

        ↓

Select Bands

        ↓

Mathematical Formula

        ↓

New Index Image

        ↓

Threshold

        ↓

Map
```

The only difference:

| Index | Purpose | Bands |
|---|---|---|
| NDVI | Vegetation | B8 + B4 |
| NDWI | Water | B3 + B8 |

---

# 2. Understand the Physics

Do not memorize only the formulas.

Remember:

## Vegetation

```text
NIR high

Red low

↓

High NDVI
```

## Water

```text
Green reflected

NIR absorbed

↓

High NDWI
```

---

# 3. NDWI Is an Image

NDWI does not give one water value.

It creates:

```text
NDWI Image

Pixel 1 → 0.6

Pixel 2 → -0.2

Pixel 3 → 0.8
```

Every pixel has its own value.

---

# Common Beginner Mistakes

## ❌ Mistake 1 — Using NDVI Bands

Wrong:

```javascript
['B8','B4']
```

This calculates NDVI.

For water:

Correct:

```javascript
['B3','B8']
```

---

## ❌ Mistake 2 — Reversing Band Order

Wrong:

```javascript
['B8','B3']
```

Correct:

```javascript
['B3','B8']
```

The order changes the sign of the result.

---

## ❌ Mistake 3 — Thinking NDWI Is One Number

NDWI is an image.

Each pixel receives its own NDWI value.

---

# Practice Exercise 🌊

Create a water detection map for another region.

Requirements:

- ✅ Choose an area containing water.
- ✅ Create a polygon.
- ✅ Load Sentinel-2.
- ✅ Display True Color.
- ✅ Calculate NDWI.
- ✅ Display NDWI.
- ✅ Create water mask:

```javascript
ndwi.gt(0)
```

- ✅ Display water only.

---

# 🧠 Check Your Understanding

Before coding, answer:

## Question 1

Why do we use:

```javascript
B3 and B8
```

for NDWI instead of:

```javascript
B4 and B8
```

---

## Question 2

Explain the difference between:

```javascript
var waterMask = ndwi.gt(0);
```

and:

```javascript
var waterMap = ndwi.updateMask(waterMask);
```

What does each variable contain?